# CBOE PUT Index Replication - Data Setup

This notebook sets up the data needed for replicating the PUT index.

## Instructions
1. Run the cells below
2. When prompted, paste your CSV data into the designated cells
3. The data will be saved locally for future use

In [ ]:
import pandas as pd
import numpy as np
from io import StringIO
from pathlib import Path

# Create data directory
DATA_DIR = Path('../data')
DATA_DIR.mkdir(exist_ok=True)

print('Ready to import data!')

## Step 1: Import ATM Implied Volatility Data

Paste your ATM IV CSV data below (replace the sample data).

**Expected format:**
- Date column
- ATM IV column (as decimal, e.g., 0.15 for 15%)

In [ ]:
# PASTE YOUR ATM IV CSV DATA BELOW (between the triple quotes)
# Include the header row

atm_iv_csv = """
Date,ATM_IV
2007-01-02,0.12
2007-01-03,0.11
"""

# Parse the data
atm_iv_df = pd.read_csv(StringIO(atm_iv_csv.strip()))
print(f"Loaded {len(atm_iv_df)} rows")
print(f"Columns: {list(atm_iv_df.columns)}")
print(f"\nFirst 5 rows:")
atm_iv_df.head()

In [ ]:
# Show what columns you have - adjust the column names in the next cell if needed
print("Your columns:", list(atm_iv_df.columns))
print("\nSample values:")
print(atm_iv_df.head(10))

In [ ]:
# ADJUST COLUMN NAMES HERE if yours are different
DATE_COL = 'Date'      # Change if your date column has a different name
IV_COL = 'ATM_IV'      # Change if your IV column has a different name

# Standardize the dataframe
atm_iv_clean = atm_iv_df[[DATE_COL, IV_COL]].copy()
atm_iv_clean.columns = ['date', 'atm_iv']
atm_iv_clean['date'] = pd.to_datetime(atm_iv_clean['date'])
atm_iv_clean = atm_iv_clean.sort_values('date').reset_index(drop=True)

# Save to CSV
atm_iv_clean.to_csv(DATA_DIR / 'atm_iv.csv', index=False)
print(f"Saved {len(atm_iv_clean)} rows to data/atm_iv.csv")
atm_iv_clean.head()

## Step 2: Import S&P 500 Price Data

Paste your S&P 500 CSV data below.

In [ ]:
# PASTE YOUR S&P 500 CSV DATA BELOW (between the triple quotes)
# Include the header row

spx_csv = """
Date,Open,High,Low,Close
2007-01-02,1418.03,1429.42,1407.86,1416.60
2007-01-03,1416.60,1421.84,1408.43,1416.92
"""

# Parse the data
spx_df = pd.read_csv(StringIO(spx_csv.strip()))
print(f"Loaded {len(spx_df)} rows")
print(f"Columns: {list(spx_df.columns)}")
spx_df.head()

In [ ]:
# ADJUST COLUMN NAMES HERE if yours are different
SPX_DATE_COL = 'Date'
SPX_CLOSE_COL = 'Close'

# Standardize
spx_clean = spx_df.copy()
spx_clean['date'] = pd.to_datetime(spx_clean[SPX_DATE_COL])
spx_clean = spx_clean.sort_values('date').reset_index(drop=True)

# Save
spx_clean.to_csv(DATA_DIR / 'spx_prices.csv', index=False)
print(f"Saved {len(spx_clean)} rows to data/spx_prices.csv")
spx_clean.head()

## Step 3: Import PUT Index Data (for validation)

Paste your official PUT index values below.

In [ ]:
# PASTE YOUR PUT INDEX CSV DATA BELOW (between the triple quotes)
# Include the header row

put_index_csv = """
Date,PUT
2007-01-02,100.00
2007-01-03,100.25
"""

# Parse the data
put_df = pd.read_csv(StringIO(put_index_csv.strip()))
print(f"Loaded {len(put_df)} rows")
print(f"Columns: {list(put_df.columns)}")
put_df.head()

In [ ]:
# ADJUST COLUMN NAMES HERE if yours are different
PUT_DATE_COL = 'Date'
PUT_VALUE_COL = 'PUT'

# Standardize
put_clean = put_df[[PUT_DATE_COL, PUT_VALUE_COL]].copy()
put_clean.columns = ['date', 'put_index']
put_clean['date'] = pd.to_datetime(put_clean['date'])
put_clean = put_clean.sort_values('date').reset_index(drop=True)

# Save
put_clean.to_csv(DATA_DIR / 'put_index.csv', index=False)
print(f"Saved {len(put_clean)} rows to data/put_index.csv")
put_clean.head()

## Step 4: Fetch T-Bill Rates from FRED

This fetches 4-week T-bill rates automatically.

In [ ]:
# Install fredapi if needed
try:
    import fredapi
except ImportError:
    !pip install fredapi
    import fredapi

# We'll use pandas_datareader as backup, or manual FRED download
try:
    import pandas_datareader as pdr
    
    # Fetch 4-week T-bill rate
    tbill = pdr.get_data_fred('DTB4WK', start='2005-01-01')
    tbill = tbill.reset_index()
    tbill.columns = ['date', 'tbill_rate']
    tbill['tbill_rate'] = tbill['tbill_rate'] / 100  # Convert to decimal
    
    tbill.to_csv(DATA_DIR / 'tbill_rates.csv', index=False)
    print(f"Saved {len(tbill)} rows to data/tbill_rates.csv")
    tbill.head()
    
except Exception as e:
    print(f"Could not fetch from FRED: {e}")
    print("\nYou can manually download from:")
    print("https://fred.stlouisfed.org/series/DTB4WK")
    print("\nOr paste T-bill data in the next cell")

In [ ]:
# BACKUP: Paste T-bill data manually if FRED fetch failed
# Only run this cell if the above failed

tbill_csv = """
Date,Rate
2007-01-02,5.03
2007-01-03,5.00
"""

# Uncomment below to use manual data:
# tbill = pd.read_csv(StringIO(tbill_csv.strip()))
# tbill.columns = ['date', 'tbill_rate']
# tbill['date'] = pd.to_datetime(tbill['date'])
# tbill['tbill_rate'] = tbill['tbill_rate'] / 100  # Convert to decimal if in %
# tbill.to_csv(DATA_DIR / 'tbill_rates.csv', index=False)
# print(f"Saved {len(tbill)} rows")

## Step 5: Verify All Data

In [ ]:
# Load all saved data and verify
print("=" * 50)
print("DATA SUMMARY")
print("=" * 50)

for f in DATA_DIR.glob('*.csv'):
    df = pd.read_csv(f)
    df['date'] = pd.to_datetime(df.iloc[:, 0])
    print(f"\n{f.name}:")
    print(f"  Rows: {len(df):,}")
    print(f"  Date range: {df['date'].min().date()} to {df['date'].max().date()}")
    print(f"  Columns: {list(df.columns)}")

## Next Steps

Once your data is loaded, proceed to:
- `02_roll_dates.ipynb` - Calculate monthly roll dates
- `03_strategy_logic.ipynb` - Implement the PUT strategy